# 04 · Deep Agent (local) — a SOC campaign coordinator you can run and audit offline

**Vertical:** Cybersecurity · **Use case:** alert-triage · **ADLC stage:** Deep Agent (capstone) ·
**No LangSmith — runs entirely from this notebook.**

Labs 01–03 built a single ReAct triage agent. This capstone promotes it to a **deep agent** — a managed
**harness** (filesystem workspace, planner, sub-agents, backends + memory, middleware, HITL, a file-based
identity + skills) — for the hard case a single agent handles poorly: correlating a **multi-alert
campaign**. We add each piece **one kwarg at a time**, then **observe it locally** and **measure** it.

![Deep agent](images/04_deep_agent.png)

| Section | Concept | Kwarg added |
|---|---|---|
| 1–3 Baseline → harness → contrast | thin `create_agent` vs the managed harness | `create_agent` → `create_deep_agent(model=…)` |
| 4 Ground it | the Lab 01 SOC tools | `tools=[…]` |
| 5 Delegate | one investigator sub-agent, isolated per alert | `subagents=[…]` + `task` |
| 6 Short-term memory | within one case (thread) | `checkpointer=` |
| 7 Long-term memory | across cases (threads) | `backend=CompositeBackend(…)` · `store=` |
| 8 Identity + skill | coordinator persona + on-demand correlation playbook | `memory=["/AGENTS.md"]` · `skills=["/skills/"]` |
| 9 Observability | **build the trace yourself** — audit log + message history + tokens | `middleware=[…]` |
| 10 HITL | approve before the agent commits a decision | `interrupt_on={…}` |
| 11 Complete agent | compose it + a typed verdict | `response_format=CampaignTriageReport` |
| 12–13 Measure | local eval loop + an honest head-to-head | *(no LangSmith)* |

> Self-directed module · ~35 min. By the end you can:
> - build a deep agent incrementally and read each new piece in the **live graph + local trace**
> - separate **short-term** (thread) memory from **long-term** (cross-thread) memory
> - give it a **file-based identity + an on-demand skill**, and see the skill load without a hosted UI
> - **observe it locally** — a SOC audit log, the full tool-call history, token/cost — no platform needed
> - **measure** the deep agent with a local eval loop, and read the head-to-head **honestly**
>
> 🧭 **Builds on Labs 01–03; runs standalone.** Data is synthetic/fictional. **No LangSmith, no hosted
> UI** — everything runs from this notebook. The full run trajectory (input → tool calls → delegation →
> output) lives in the returned `result`, and we render it right here.

## 0. Setup

Thin setup: load `.env`, keep the model-gateway hygiene, and put the shared `utils/` + the use case's
graph module on the path. **No LangSmith workspace print and no experiment-reset block** — this lab
produces no traced runs, so there is nothing to route or clean up.

In [ ]:
import json
import sys
import os
from collections import defaultdict
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # FIRST — before any model client is constructed
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))  # so `utils` + the graph module import; _root is the `content/` dir

One model powers everything (agents and judges). `get_model()` reads the provider key from the
environment — **never hardcoded** — and follows the `LABS_MODEL` override, so the whole notebook swaps
providers in one place.

In [ ]:
from utils.models import DEFAULT_MODEL, get_model

model = get_model()
print("model:", DEFAULT_MODEL)

We reuse Lab 01's grounding tools, its system prompt, and the campaign schema as **plumbing** — imported
from the use case's graph module, not rebuilt here (they were the lesson in `01_build`). We read the
prompt straight from disk so there is **no LangSmith Prompt-Hub call**, even if a key happens to be set.

In [ ]:
from alert_triage_graph import (
    AGENT_DIR,             # data/agent/ — holds AGENTS.md + skills/ for the coordinator
    CampaignTriageReport,  # the typed campaign verdict (Pydantic) — defined in 01_build's graph module
    PROMPT_PATH,           # data/prompt.md — the SOC analyst system prompt
    get_alert,
    get_asset_context,
    search_incident_history,
    search_soc_kb,
)

TRIAGE_TOOLS = [get_alert, get_asset_context, search_incident_history, search_soc_kb]
base_prompt = PROMPT_PATH.read_text().strip()  # local read — no Prompt Hub, no network
DATASET_PATH = _root / "evals" / "dataset_alert_triage_campaign.json"  # committed synthetic eval set


def last(result) -> str:
    """The final message text of an invoke result — a tiny convenience used across run cells."""
    return result["messages"][-1].content

## 1. The baseline — a plain `create_agent`

`create_agent(model, tools=[])` is the Labs 01–03 shape: a model in a tool-calling loop. Run it, then
note its **thin** shape — one model call, no middleware. Hold that in mind; the harness is the contrast.

In [ ]:
from langchain.agents import create_agent

simple = create_agent(model=model, tools=[])  # no tools — the plainest agent
simple_out = simple.invoke({"messages": [{"role": "user", "content":
    "In one sentence, what does a SOC analyst mean by 'triaging' a security alert?"}]})
print(last(simple_out)[:300])

In [ ]:
from IPython.display import Image, display

display(Image(simple.get_graph().draw_mermaid_png()))

## 2. The deep-agent harness — `create_deep_agent`

Same model, still **no tools** — but `create_deep_agent(model=…)` already ships a working **harness**:
filesystem tools, a `write_todos` planner, the `task` delegation tool, and automatic context management.

In [ ]:
from deepagents import create_deep_agent

harness = create_deep_agent(model=model)  # no domain tools, no sub-agents — just the harness


def show_graph(agent, fallback_png: str | None = None) -> None:
    """Render the graph inline; deep-agent node names (brackets/dots from middleware/sub-agents) break the
    remote mermaid.ink renderer, so fall back to a pre-rendered brand PNG, else list the node names."""
    graph = agent.get_graph()
    try:
        display(Image(graph.draw_mermaid_png()))
    except Exception as exc:  # 400 on middleware/sub-agent node names
        if fallback_png and Path(fallback_png).exists():
            print(f"(live render unavailable — {type(exc).__name__}; showing the pre-rendered architecture)")
            display(Image(filename=fallback_png))
        else:
            print(f"(live diagram unavailable: {type(exc).__name__}) — nodes:",
                  ", ".join(sorted(str(n) for n in graph.nodes)))

The harness can already keep a **triage scratchpad** — write intermediate notes to a file and read them
back — with no tool we wrote.

In [ ]:
harness_out = harness.invoke({"messages": [{"role": "user", "content":
    "Write 'working notes: checking ALT-2003' to /scratch.md, then read it back and confirm the contents."}]})
print(last(harness_out)[:300])

What the harness ships for free — a single ReAct agent (Labs 01–03) has **none** of this:

| Built-in | What it is |
|---|---|
| Filesystem tools | `ls` · `read_file` · `write_file` · `edit_file` · `glob` · `grep` — a scratchpad |
| Planner | `write_todos` — plan multi-step work |
| Delegation | `task` — hand work to sub-agents (§5) |
| Context management | auto-summarize near the limit + offload large tool results to files |

## 3. See the contrast — render it locally

The lesson is the **difference** between the two. Print each agent's own message history with
`.pretty_print()` and read it right here: the plain agent is a single model turn; the harness is a stack
of **built-in middleware** doing filesystem + planning work.

> 💡 This local side-by-side **is** your trace — the turns, the tool calls, and the tokens are all in
> `result["messages"]`; we just render them in-notebook.

In [ ]:
print("── PLAIN create_agent — thin: one model turn ───────────────")
for m in simple_out["messages"]:
    m.pretty_print()

In [ ]:
print("── create_deep_agent HARNESS — filesystem + planner middleware at work ──")
for m in harness_out["messages"]:
    m.pretty_print()

## 4. Ground it — the SOC tools

The harness can plan and take notes, but it can't see SOC data yet. Add Lab 01's grounding tools —
imported as plumbing, passed with one `tools=` kwarg — and it can triage a real alert.

In [ ]:
triage_agent = create_deep_agent(
    model=model,
    system_prompt=base_prompt,
    tools=TRIAGE_TOOLS,  # ← only addition vs §2: the SOC grounding tools
)

triage_out = triage_agent.invoke({"messages": [{"role": "user", "content":
    "Triage ALT-2003 and give a one-line disposition with citations."}]})
print(last(triage_out)[:400])

> 💡 One grounded agent triages **one** alert well. A **campaign** is many alerts plus a correlation —
> and holding every alert's tool output in one context is exactly where a single agent gets muddled.
> Delegation fixes that next.

## 5. Delegate — one investigator sub-agent

Rather than triage every alert in one context, delegate each to a scoped **sub-agent** and let a
**coordinator** correlate. A sub-agent is a plain dict; its `description` is what the coordinator reads
to decide when to delegate. **One** definition, invoked once per alert.

![Coordinator + sub-agent](images/04_coordinator.png)

In [ ]:
investigator = {
    "name": "alert-investigator",
    "description": (
        "Fully investigate ONE alert: retrieve it, enrich every asset, and match prior incidents + SOC "
        "guidance. Report cited evidence and the accounts/IPs/hosts/domains it touches. Delegate to this "
        "sub-agent once per alert, naming the alert. Do not decide the campaign."
    ),
    "system_prompt": (
        "You investigate exactly one alert end to end. Use get_alert, then get_asset_context for every "
        "asset, then search_incident_history and search_soc_kb. Report cited evidence and the entities "
        "(accounts, source IPs, hosts, domains) it involves. Do not decide whether alerts form a "
        "campaign — that is the coordinator's job."
    ),
    "tools": TRIAGE_TOOLS,  # the leaf tools live on the sub-agent, not the coordinator
}

The coordinator holds **no leaf tools** — only the built-in `task`. That forces it to *delegate* each
alert (isolated context) and then correlate, instead of triaging inline.

In [ ]:
coordinator_prompt = (
    base_prompt
    + "\n\nYou coordinate multi-alert campaign investigations. Delegate each alert to the "
    "alert-investigator sub-agent (once per alert, naming it), correlate the findings on shared entities, "
    "exclude look-alike decoys, and synthesize one verdict. Do not invent facts absent from sub-agent results."
)

coordinator = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],  # ← only addition vs §4: one sub-agent (coordinator holds only `task`)
)

show_graph(coordinator, fallback_png="images/04_coordinator.png")

**Watch the delegation fire — live, locally.** `stream_mode="updates"` yields each node's update as it
happens, so you see one `task` call per alert stream past. This is the offline equivalent of watching a
trace populate.

In [ ]:
def stream_tool_calls(agent, question: str, config: dict | None = None) -> dict:
    """Stream a run and print each tool call as its node fires; return the final result dict."""
    final: dict = {}
    for chunk in agent.stream({"messages": [{"role": "user", "content": question}]},
                              config=config, stream_mode="updates"):
        for node, update in (chunk or {}).items():
            for m in (update or {}).get("messages", []) if isinstance(update, dict) else []:
                for tc in getattr(m, "tool_calls", None) or []:
                    arg = next(iter(tc.get("args", {}).values()), "")
                    print(f"  → {node}: {tc['name']}({str(arg)[:48]})")
                final = update
    return final


CAMPAIGN_ALERTS = ["ALT-2004", "ALT-2005", "ALT-2006", "ALT-2007", "ALT-2009"]  # 4 linked + 1 decoy
print("delegations as they fire:")
_ = stream_tool_calls(coordinator,
    "Triage these alerts together as one investigation: " + ", ".join(CAMPAIGN_ALERTS) +
    ". Decide if they are a coordinated campaign, exclude any decoy, and give shared pivots + a verdict.")

> 💡 Each `task(...)` line is a labeled sub-investigation in an isolated context — the same signal a
> hosted trace's nested tree shows, printed here as it happens. We **count** these in §12 as the proof
> the work was decomposed.

## 6. Short-term memory — within one case (a checkpointer)

A `checkpointer` persists conversation state by `thread_id`. A follow-up in the **same** thread
remembers; a **new** thread starts fresh. This is short-term memory — in-memory, per-case.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

short_term = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    checkpointer=InMemorySaver(),  # ← only addition vs §5: thread-scoped memory
)

case_1 = {"configurable": {"thread_id": "case-1"}}
short_term.invoke({"messages": [{"role": "user", "content": "I'm working case CASE-42 about ALT-2004."}]}, config=case_1)
same = short_term.invoke({"messages": [{"role": "user", "content":
    "Which case and alert did I just mention? Answer in one line."}]}, config=case_1)  # SAME thread → remembers
print("same thread (remembers):", last(same)[:160])

case_2 = {"configurable": {"thread_id": "case-2"}}
fresh = short_term.invoke({"messages": [{"role": "user", "content":
    "Which case and alert did I just mention? Answer in one line."}]}, config=case_2)  # NEW thread → forgets
print("new thread  (forgets) :", last(fresh)[:160])

> 💡 Short-term memory lives in the checkpointer, keyed by `thread_id` — remembered **within** a case,
> gone in a **new** one. To carry knowledge **across** cases you need a durable store — next.

## 7. Long-term memory — across cases (a durable store)

Add a durable store and route only `/memories/` to it with a `CompositeBackend`. Now a note written in
one case (thread) is recalled in **another** — the cross-thread memory a checkpointer alone can't give.

![Backends & memory](images/04_backends.png)

| Backend | Scope |
|---|---|
| `StateBackend` (default) | ephemeral — files vanish when the thread ends |
| `StoreBackend` (+ a `store`) | durable key-value, survives across threads |
| `CompositeBackend(default=…, routes={…})` | route some paths (e.g. `/memories/`) to the durable store |

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

MEMORY_NS = ("meridian-union", "soc-memories")  # scope the durable memory (here: one synthetic tenant)
durable_store = InMemoryStore()
memory_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda rt: MEMORY_NS)},  # only /memories/ is durable
)

memory_agent = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    backend=memory_backend,        # ← addition vs §6: a durable /memories/ route
    store=durable_store,           # backing store; swap for a Postgres store in production
    checkpointer=InMemorySaver(),  # keep short-term thread continuity too
)

In [ ]:
thread_a = {"configurable": {"thread_id": "case-write"}}
thread_b = {"configurable": {"thread_id": "case-read"}}
memory_agent.invoke({"messages": [{"role": "user", "content":
    "Save this to /memories/prior.md exactly: 'acct-fin-controller-x was involved in a prior finance campaign.'"}]},
    config=thread_a)
recall = memory_agent.invoke({"messages": [{"role": "user", "content":
    "Read /memories/prior.md. What account does it flag? One line."}]}, config=thread_b)  # different thread!
print("new thread, durable recall:", last(recall)[:200])

> 💡 The checkpointer is *short-term* (one thread); the routed `StoreBackend` is *long-term* (across
> threads). Same-thread remembers → new-thread forgets → durable memory survives both.

## 8. File-based identity + a skill

Instead of one giant system prompt, a deep agent loads an editable **`AGENTS.md`** (identity, read at
startup) and **skills** (`SKILL.md`, loaded **on demand** when relevant). We serve both from a
`FilesystemBackend` rooted at the use case's `data/agent/` folder. This is **required** in a capstone —
skills are how the agent gains reusable expertise without bloating every prompt.

In [ ]:
print("=== AGENTS.md (always loaded — the coordinator's identity) ===")
print((AGENT_DIR / "AGENTS.md").read_text()[:600], "…\n")
print("=== skills/campaign-correlation/SKILL.md (loaded on demand) ===")
print((AGENT_DIR / "skills" / "campaign-correlation" / "SKILL.md").read_text()[:500], "…")

In [ ]:
from deepagents.backends import FilesystemBackend

identity_agent = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    backend=FilesystemBackend(root_dir=str(AGENT_DIR), virtual_mode=True),  # serves AGENTS.md + skills/
    memory=["/AGENTS.md"],  # ← identity, always loaded
    skills=["/skills/"],    # ← on-demand campaign-correlation skill
)

Ask a question that should trigger the correlation skill, and stream the run: you'll see the agent
**read the skill file on demand** (a filesystem call) before correlating — the progressive-disclosure
load, visible locally with no hosted trace.

In [ ]:
_ = stream_tool_calls(identity_agent,
    "Using your campaign-correlation skill, triage ALT-2004, ALT-2005, ALT-2006 together and name the shared pivot.")

> 💡 **Memory vs skills.** `memory=["/AGENTS.md"]` is *always* loaded (identity). `skills=["/skills/"]`
> are loaded *only when relevant* — cheaper context, and each skill evolves as its own file without
> touching the prompt.

## 9. Observability
| Lens | What it gives a SOC |
|---|---|
| audit-log **middleware** (`@wrap_tool_call`) | every tool call recorded — even inside sub-agents |
| `.pretty_print()` over `result["messages"]` | the full input→output trajectory (the hand-built trace) |
| `usage_metadata` on AI messages | tokens (→ cost) per run |

### Audit-log middleware

Middleware wraps every tool call. A `@wrap_tool_call` hook appends the **tool name** to an audit list
before running the tool — the SOC's tamper-evident record of what the agent did. (Log the name only; in
production you'd redact args with `PIIMiddleware` before persisting — see the healthcare lab.)

In [ ]:
from langchain.agents.middleware import wrap_tool_call

AUDIT: list[str] = []


@wrap_tool_call
def audit_log(request, handler):
    call = getattr(request, "tool_call", None) or {}
    AUDIT.append(call.get("name", "?") if isinstance(call, dict) else str(call))
    return handler(request)  # then run the tool as normal


audited = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    middleware=[audit_log],  # ← addition: an audit log of every tool call
)
audited_result = audited.invoke({"messages": [{"role": "user", "content":
    "Triage ALT-2004 and ALT-2005 together; name the shared source IP."}]})
print("audit log (tool calls, in order):", AUDIT)

### The full trajectory — `.pretty_print()`

The hosted trace's job is to show input → tool calls → delegation → output. That whole trajectory is in
`result["messages"]`; print it. (Truncated to the first turns so the cell stays readable.)

In [ ]:
for m in audited_result["messages"][:6]:
    m.pretty_print()

### Tokens & cost — `usage_metadata`

Every AI message carries `usage_metadata`. Sum it for a per-run token count — the number a SOC lead
watches when a triage agent runs thousands of times a day.

In [ ]:
def token_usage(result: dict) -> dict:
    totals: dict = defaultdict(int)
    for m in result["messages"]:
        for k, v in (getattr(m, "usage_metadata", None) or {}).items():
            if isinstance(v, int):
                totals[k] += v
    return dict(totals)


print("tokens this run:", token_usage(audited_result))

## 10. Human-in-the-loop — approve before committing a decision

`interrupt_on` pauses the agent before a chosen tool so a human can approve. In a SOC you gate the
moment the agent **commits a decision to the case file** (a `write_file`): it pauses, you approve, it
resumes. HITL needs persistence, so a checkpointer is required.

`invoke → ⏸️ interrupt → decide → Command(resume=…) → ▶️ continue`

In [ ]:
from langgraph.types import Command

approver = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    interrupt_on={"write_file": True},  # ← pause before any write to the case file
    checkpointer=InMemorySaver(),       # required for HITL
)
hitl_cfg = {"configurable": {"thread_id": "hitl-demo"}}
paused = approver.invoke({"messages": [{"role": "user", "content":
    "Triage ALT-2004, then write your one-line decision to /case-notes/ALT-2004.md."}]}, config=hitl_cfg)
print("⏸️ interrupted:", "__interrupt__" in paused)

**Reviewer approves → resume.** In production a person decides; headless, we approve so the notebook runs.

In [ ]:
resumed = approver.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=hitl_cfg)  # ▶️ approve
print("▶️ completed:", "__interrupt__" not in resumed)

## 11. The complete campaign agent

Assemble it: coordinator + one investigator, `AGENTS.md` identity + skill from a `FilesystemBackend`, and
a typed **`CampaignTriageReport`** (linked alerts, decoys, pivots, aggregate severity — the schema is
code, imported from Lab 01). Memory, middleware, and HITL from §6–10 compose in the same way when a
deployment needs them.

![Complete agent](images/04_complete.png)

In [ ]:
deep_agent = create_deep_agent(
    model=model,
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    backend=FilesystemBackend(root_dir=str(AGENT_DIR), virtual_mode=True),
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    response_format=CampaignTriageReport,  # ← typed campaign verdict
)

show_graph(deep_agent, fallback_png="images/04_complete.png")

Run the cluster — four linked alerts (an account-takeover kill-chain on one source IP) plus a look-alike
decoy — and read the **typed** verdict.

In [ ]:
campaign_q = (
    "Triage these five alerts together as one investigation: " + ", ".join(CAMPAIGN_ALERTS) + ". "
    "Decide whether they are a coordinated campaign, exclude any decoy, and give the shared pivots, an "
    "aggregate severity/escalation, and one containment plan."
)
final = deep_agent.invoke({"messages": [{"role": "user", "content": campaign_q}]})
report = final["structured_response"]

print(f"coordinated campaign : {report.is_coordinated_campaign}")
print(f"aggregate severity   : {report.aggregate_severity}  | escalate: {report.aggregate_escalate}")
print(f"linked alerts        : {', '.join(report.linked_alert_ids)}")
print(f"excluded (decoys)    : {', '.join(report.excluded_alert_ids)}")
print(f"pivot entities       : {', '.join(report.pivot_entities)}")
print("citations           :", ", ".join(report.citations))

## 12. Measure it — a local eval loop

Tutorials *assert* a deep agent's value; we **measure** it — with no LangSmith. We read the committed
synthetic campaign dataset from disk and score each example with **three** evaluators (the fixed
vocabulary):

| Evaluator | Kind | Checks |
|---|---|---|
| `correctness` | reference-based LLM judge | linked set, decoy exclusion, aggregate severity/escalation |
| `tool_calls` | deterministic | how many tool calls the coordinator made (from `result["messages"]`) |
| `subagent_delegations` | deterministic | `task` hand-offs — the deep-agent signal earlier labs lack |

In [ ]:
examples = json.loads(DATASET_PATH.read_text())
print("examples:", len(examples))
example = examples[0]
print("Q         :", example["inputs"]["messages"][-1]["content"][:150], "…")
print("reference :", example["outputs"]["reference_answer"][:240], "…")

### The eval target

Reuse the **§11 deep agent** — we grade what a deployment would run. Copy the input messages: a deep
agent normalizes them in place, which would otherwise mutate the shared eval example.

In [ ]:
def run_deep(inputs: dict) -> dict:
    messages = [dict(m) for m in inputs["messages"]]  # copy — deep agents mutate messages in place
    result = deep_agent.invoke({"messages": messages})
    return {"answer": result["structured_response"].model_dump_json(indent=2), "messages": result["messages"]}

### Evaluator #1: `correctness` (LLM-as-judge)

Reference-based. The full judge prompt is a **visible constant** — the correlation nuance lives here, in
the prompt, so the *key* stays the standard `correctness`. It passes correct extra detail and fails only
on a wrong linked set, a decoy left in, a missed campaign call, or a wrong aggregate severity.

In [ ]:
from pydantic import BaseModel, Field

CORRECTNESS_PROMPT = """You grade a SOC campaign-triage answer against a reference correlation.

Return True only if the answer gets the CAMPAIGN decision right: the coordinated-vs-independent call,
the set of linked alert IDs, the excluded decoys, and the aggregate severity/escalation. Allow correct
extra detail beyond the reference. Fail ONLY for a contradiction: a wrong linked set, a decoy left in,
missing the coordinated call, or a wrong aggregate severity/escalation.

<question>{question}</question>
<answer>{answer}</answer>
<reference_answer>{reference}</reference_answer>
<reference_is_coordinated>{coordinated}</reference_is_coordinated>
<reference_linked>{linked}</reference_linked>
<reference_excluded>{excluded}</reference_excluded>
<reference_aggregate_severity>{severity}</reference_aggregate_severity>"""


class CorrectnessScore(BaseModel):
    reasoning: str = Field(description="Which correlation elements matched or missed.")
    score: bool = Field(description="True if the campaign decision is correct.")


def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judge = get_model().with_structured_output(CorrectnessScore)
    v = judge.invoke(CORRECTNESS_PROMPT.format(
        question=inputs["messages"][-1]["content"],
        answer=outputs["answer"],
        reference=reference_outputs["reference_answer"],
        coordinated=reference_outputs["is_coordinated_campaign"],
        linked=", ".join(reference_outputs["linked_alert_ids"]) or "none",
        excluded=", ".join(reference_outputs["excluded_alert_ids"]) or "none",
        severity=reference_outputs["aggregate_severity"],
    ))
    return {"key": "correctness", "score": v.score, "comment": v.reasoning}

### Evaluators #2 & #3: `tool_calls` and `subagent_delegations` (deterministic)

No LLM — walk `result["messages"]` and count `AIMessage.tool_calls`. `subagent_delegations` counts only
the `task` hand-offs; a single agent scores **0** there structurally, which is the whole point of §13.

In [ ]:
def _count_tool_calls(messages, name: str | None = None) -> int:
    return sum(
        1
        for m in messages
        for tc in (getattr(m, "tool_calls", None) or [])
        if name is None or tc.get("name") == name
    )


def tool_calls_evaluator(outputs: dict) -> dict:
    return {"key": "tool_calls", "score": _count_tool_calls(outputs["messages"]), "comment": "coordinator tool calls."}


def subagent_delegations_evaluator(outputs: dict) -> dict:
    n = _count_tool_calls(outputs["messages"], name="task")
    return {"key": "subagent_delegations", "score": n, "comment": "task hand-offs in the message history."}

### Dry-run one example first

In [ ]:
dry = run_deep(example["inputs"])
print("correctness         :", correctness_evaluator(example["inputs"], dry, example["outputs"]))
print("tool_calls          :", tool_calls_evaluator(dry))
print("subagent_delegations:", subagent_delegations_evaluator(dry))

### Scale to the dataset → scorecard

The local eval loop replaces LangSmith's `evaluate()`: iterate the dataset, run the agent once per
example, score, and print an aggregated scorecard — ~15 fully-visible lines, no hosted experiment.

In [ ]:
scores: dict[str, list] = defaultdict(list)
for ex in examples:
    out = run_deep(ex["inputs"])
    scores["correctness"].append(correctness_evaluator(ex["inputs"], out, ex["outputs"])["score"])
    scores["tool_calls"].append(tool_calls_evaluator(out)["score"])
    scores["subagent_delegations"].append(subagent_delegations_evaluator(out)["score"])

print("\n📊 Scorecard — deep agent (local)")
for key, vals in scores.items():
    if vals and all(isinstance(v, bool) for v in vals):
        print(f"  {key:22s}: {sum(vals)}/{len(vals)} passed")
    else:
        print(f"  {key:22s}: avg {sum(vals) / len(vals):.1f}")

## 13. The honest head-to-head — single agent vs deep agent

Does delegation earn its keep? We test **architecture, not prompt**: a single `create_agent` gets the
**same** tools, the **same** typed output, and **matched** correlation instructions — it just holds every
alert in one context instead of delegating. Then we run both on the same campaigns.

In [ ]:
single_prompt = (
    base_prompt
    + "\n\nYou triage multi-alert campaigns in ONE pass: investigate each alert with your tools, then "
    "correlate on shared entities, exclude look-alike decoys, and synthesize one verdict. Do not invent "
    "facts."  # matched to the coordinator's goal — the only difference is no delegation
)
single_agent = create_agent(
    model=model,
    tools=TRIAGE_TOOLS,                     # same tools — but all in one context
    system_prompt=single_prompt,
    response_format=CampaignTriageReport,   # same typed output
)


def run_single(inputs: dict) -> dict:
    result = single_agent.invoke({"messages": [dict(m) for m in inputs["messages"]]})
    return {"answer": result["structured_response"].model_dump_json(indent=2), "messages": result["messages"]}

In [ ]:
print(f"{'':22s}{'single':>10}{'deep':>10}")
head_to_head: dict[str, dict[str, list]] = {"single": defaultdict(list), "deep": defaultdict(list)}
for ex in examples:
    for label, runner in (("single", run_single), ("deep", run_deep)):
        out = runner(ex["inputs"])
        head_to_head[label]["correctness"].append(correctness_evaluator(ex["inputs"], out, ex["outputs"])["score"])
        head_to_head[label]["subagent_delegations"].append(subagent_delegations_evaluator(out)["score"])

for key in ("correctness", "subagent_delegations"):
    s, d = head_to_head["single"][key], head_to_head["deep"][key]
    fmt = (lambda v: f"{sum(v)}/{len(v)}") if key == "correctness" else (lambda v: f"{sum(v)}")
    print(f"{key:22s}{fmt(s):>10}{fmt(d):>10}")

**Read it honestly.** On a handful of alerts a capable model correlates about as well in one context, so
`correctness` usually **ties** — and that's fine. The difference that matters is structural:

| Signal | Single | Deep | Why it matters |
|---|---|---|---|
| `subagent_delegations` | **0** | **> 0** | each alert is a traceable, isolated sub-investigation |
| planner / todos, context hygiene | — | ✓ | scales past the point where one flat context degrades |
| `AGENTS.md` + skills | — | ✓ | identity + correlation expertise evolve as files, not prompt edits |

> 💡 **The capstone lesson:** a deep agent is a **harness**, not a scoreboard trick. Reach for it when
> you need legibility, memory, approval gates, and scale — then **measure** to confirm it earns its keep.
> Delegation didn't lose here; it bought structure the single agent structurally cannot show.

## 14. Recap & next

| Topic | API |
|---|---|
| Deep-agent harness | `create_deep_agent(model=…)` → filesystem tools + `write_todos` + context mgmt |
| Sub-agent delegation | `subagents=[{…}]` + built-in `task` (coordinator holds no leaf tools) |
| Short-term vs long-term memory | `checkpointer=` · `CompositeBackend(routes={"/memories/": StoreBackend(…)})` |
| Identity + skills | `memory=["/AGENTS.md"]`, `skills=["/skills/"]` on a `FilesystemBackend` |
| Local observability | `@wrap_tool_call` audit log · `.pretty_print()` · `usage_metadata` · `stream_mode="updates"` |
| HITL | `interrupt_on={"write_file": True}` + `Command(resume=…)` |
| Typed output + local eval | `response_format=CampaignTriageReport` + a local loop over the committed dataset |

The arc: **built** (01) → **evaluated** (02) → **orchestrated** (03) → **deep-agent harness** (04) — and
everything here ran with **no LangSmith**. Swap the synthetic fixtures for a customer's SOC corpus and
the shape holds. The same coordinator is packaged as `build_deep_agent_v2` in `alert_triage_graph.py`
for deployment (e.g. `langgraph dev`); no hosted Studio is required to run this notebook.

**Next:** re-enable LangSmith when you have platform access to add hosted traces, datasets, online evals,
and Studio — everything you built locally here maps onto those one-to-one.